# qq.ipynb

`tog.ipynb` 기반의 빠른 헬멧 판정 + 단일 최종 라벨 개선판입니다.

중점 수정:

1. Roboflow hosted 헬멧 판정은 영상당 호출 수를 제한해서 멈춘 것처럼 보이는 현상을 줄임
2. 로컬 헬멧 `best.pt`가 있으면 Roboflow보다 로컬 YOLO를 우선 사용
3. 한 영상에 여러 이상행동 태그를 최종 라벨로 붙이지 않고, 후보 점수 중 가장 강한 것을 `final_label`로 선택
4. 후보군 전체는 `candidate_scores`, `candidate_tags`에 남겨서 확인 가능
5. 결과를 영상별 표 형태로 보기 쉽게 출력


In [1]:
# ============================================================
# Cell 1. 경로 및 전체 설정  [qq 개선판]
# ============================================================
from pathlib import Path
import os, json, math, time, gc, warnings, tempfile, shutil, subprocess, sys
from collections import Counter, defaultdict, deque

AIO_VERSION = "qq_fast_helmet_primary_label_v1"

VIDEO_ROOT = Path(r"C:\Users\우성한\Sample (1)\Sample\01.원천데이터")
OUTPUT_ROOT = Path(r"C:\Users\우성한\outputs\prediction")

AIO_DIR = OUTPUT_ROOT / "qq"
AIO_FEATURE_DIR = AIO_DIR / "aio_features"
AIO_FINAL_DIR = AIO_DIR / "final_aio"
AIO_DEBUG_DIR = AIO_DIR / "debug"
HELMET_CROP_DIR = AIO_DEBUG_DIR / "helmet_crops"

for d in [AIO_DIR, AIO_FEATURE_DIR, AIO_FINAL_DIR, AIO_DEBUG_DIR, HELMET_CROP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

VIDEO_EXTS = {
    ".mp4", ".avi", ".mov", ".mkv", ".wmv", ".mpeg", ".mpg", ".m4v",
    ".MP4", ".AVI", ".MOV", ".MKV", ".WMV", ".MPEG", ".MPG", ".M4V"
}

# 전체 처리 시 None. 테스트 시 2~5 정도로 제한 가능.
MAX_VIDEOS = None
MAX_FRAMES_PER_VIDEO = None

# 새 코드 구조에서는 기존 feature JSON을 덮어쓰는 것이 안전하다.
OVERWRITE_FEATURES = True

# 샘플링 설정
FRAME_INTERVAL = 10
WINDOW_SECONDS = 5.0
WINDOW_STRIDE_SECONDS = 1.5

# 모델 설정
BASE_YOLO_MODEL_PATH = "yolov8n.pt"
POSE_MODEL_CANDIDATES = ["yolo11n-pose.pt", "yolov8n-pose.pt"]
YOLO_WORLD_MODEL_CANDIDATES = ["yolov8s-world.pt", "yolov8m-world.pt"]

# SegFormer 설정: CUDA면 b5, CPU면 b0 fallback
SEG_MODEL_ID_GPU = "nvidia/segformer-b5-finetuned-cityscapes-1024-1024"
SEG_MODEL_ID_CPU = "nvidia/segformer-b0-finetuned-cityscapes-1024-1024"

# Cityscapes label 기준
ROAD_ID = 0
SIDEWALK_ID = 1

# threshold: 결과가 너무 포괄적이면 올리고, 너무 놓치면 내리면 됨
FIGHT_THRESHOLD = 0.44
STALKING_THRESHOLD = 0.42
RIDE_SIDEWALK_THRESHOLD = 0.30
RIDE_SIDEWALK_CANDIDATE_THRESHOLD = 0.18
NO_HELMET_THRESHOLD = 0.30

# 검출 confidence
PERSON_MIN_CONF = 0.18
VEHICLE_MIN_CONF = 0.12
WORLD_MIN_CONF = 0.05
POSE_MIN_CONF = 0.15

# 트래킹/행동 기준
USE_YOLO_TRACKER = True
TRACKER_YAML = "bytetrack.yaml"     # Ultralytics 내장 tracker. 안 되면 자동 fallback
TRACK_MAX_DIST = 150
TRACK_MAX_MISSED = 2
MIN_TRACK_LEN_FOR_STALKING = 5
FOLLOW_DISTANCE_MIN = 40
FOLLOW_DISTANCE_MAX = 450
FOLLOW_DIRECTION_COS_THRESHOLD = 0.55
FOLLOW_DISTANCE_STABILITY = 130

# optical flow 기준
FLOW_RESIZE_WIDTH = 480
FLOW_MAG_STRONG = 3.0

# 주행/침범 기준
TWO_WHEELER_CLASSES = {"bicycle", "motorcycle"}
WORLD_TWO_WHEELER_HINTS = {
    "bicycle", "motorcycle", "motorbike", "bike", "scooter", "electric scooter",
    "kickboard", "e-scooter", "moped", "e-bike", "electric bike"
}
RIDER_CLASSES = {"person"}
SIDEWALK_PATCH_RADIUS = 18
BOTTOM_POINT_Y_RATIO = 0.92
BOTTOM_BAND_Y1 = 0.70
BOTTOM_BAND_Y2 = 1.00
BOTTOM_BAND_X_MARGIN = 0.10

# 헬멧 모델 연결: 직접 학습한 best.pt가 있으면 여기에 넣는다.
# 예: HELMET_MODEL_PATH = r"C:\Users\우성한\outputs\prediction\runs_cf6_helmet\bike_helmet_yolov8\weights\best.pt"
HELMET_MODEL_PATH = None
AUTO_DISCOVER_HELMET_MODEL = True
HELMET_CONF = 0.15
SAVE_HELMET_CROPS = False
MAX_SAVED_HELMET_CROPS = 300

# Roboflow hosted inference 선택.
# ROBOFLOW_API_KEY 환경변수가 있으면 자동으로 켜진다.
# 단, 로컬 HELMET_MODEL_PATH가 발견되면 로컬 YOLO를 우선 사용한다.
ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "")
USE_ROBOFLOW_HOSTED_INFERENCE = bool(ROBOFLOW_API_KEY)
ROBOFLOW_MODEL_ID = "bike-helmet-detection-2vdjo/1"

# 헬멧 hosted inference 속도 제어
# FRAME_INTERVAL=10일 때 HELMET_CHECK_SAMPLE_STRIDE=3이면 원본 기준 약 30프레임마다 검사한다.
HELMET_ENABLE = True
HELMET_CHECK_SAMPLE_STRIDE = 3
MAX_HELMET_CHECKS_PER_VIDEO = 16
MAX_HELMET_PAIRS_PER_FRAME = 1
HELMET_EARLY_STOP_NO_HELMET_HITS = 2
HELMET_EARLY_STOP_CONFIDENT_HITS = 5

# 최종 라벨은 후보 중 가장 높은 점수 하나만 선택한다.
PRIMARY_LABEL_MIN_SCORE = 0.30
PRIMARY_LABEL_MARGIN = 0.08

# Roboflow 데이터셋 다운로드/학습용 설정. 선택 실행 셀에서 사용.
RF_WORKSPACE = "bike-helmets"
RF_PROJECT = "bike-helmet-detection-2vdjo"
RF_VERSION = 1
RF_EXPORT_FORMAT = "yolov8"
RF_DATASET_DIR = AIO_DIR / "roboflow_datasets"
RF_TRAIN_PROJECT_DIR = AIO_DIR / "runs_helmet"

print("AIO_VERSION:", AIO_VERSION)
print("VIDEO_ROOT exists:", VIDEO_ROOT.exists(), VIDEO_ROOT)
print("AIO_DIR:", AIO_DIR)

if not VIDEO_ROOT.exists():
    raise FileNotFoundError(f"원본 영상 루트를 찾지 못했습니다: {VIDEO_ROOT}")


AIO_VERSION: qq_fast_helmet_primary_label_v1
VIDEO_ROOT exists: True C:\Users\우성한\Sample (1)\Sample\01.원천데이터
AIO_DIR: C:\Users\우성한\outputs\prediction\qq


In [2]:
# ============================================================
# Cell 2. 의존성 확인 및 모델 로드  [YOLO-World/Tracker/Helmet 상태 명확화]
# ============================================================
missing = []

try:
    import cv2
    print("cv2 OK")
except Exception as e:
    missing.append("opencv-python")
    print("cv2 import failed:", repr(e))

try:
    import numpy as np
    print("numpy OK")
except Exception as e:
    missing.append("numpy")
    print("numpy import failed:", repr(e))

try:
    from tqdm import tqdm
    print("tqdm OK")
except Exception as e:
    missing.append("tqdm")
    print("tqdm import failed:", repr(e))

try:
    import torch
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("torch OK, cuda:", torch.cuda.is_available(), "device:", DEVICE)
    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = True
        print("gpu:", torch.cuda.get_device_name(0))
except Exception as e:
    torch = None
    DEVICE = "cpu"
    missing.append("torch")
    print("torch import failed:", repr(e))

try:
    from ultralytics import YOLO
    print("ultralytics YOLO OK")
except Exception as e:
    YOLO = None
    missing.append("ultralytics")
    print("ultralytics import failed:", repr(e))

try:
    from PIL import Image
    print("PIL OK")
except Exception as e:
    missing.append("pillow")
    print("PIL import failed:", repr(e))

# ByteTrack은 lap 패키지를 요구한다. 없으면 Ultralytics가 자동 설치를 시도하지만,
# 설치 직후에는 커널 재시작이 필요할 수 있다.
try:
    import lap
    LAP_STATUS = {"lap_import": True, "error": None}
    print("lap OK")
except Exception as e:
    LAP_STATUS = {"lap_import": False, "error": repr(e)}
    print("lap import failed. ByteTrack 첫 실행 때 자동 설치될 수 있습니다:", repr(e))
    print("권장: python -m pip install -U lap 후 Jupyter 커널 재시작")


if missing:
    print("\n설치 필요 후보:", sorted(set(missing)))
    print("예: pip install opencv-python numpy tqdm ultralytics torch transformers pillow roboflow inference-sdk")

if YOLO is None:
    raise RuntimeError("ultralytics YOLO를 사용할 수 없습니다.")

# ------------------------------------------------------------
# YOLO-World의 set_classes는 clip 의존성이 필요하다.
# Git/clip이 없으면 world_model을 '비활성'으로 표시하고 기본 YOLO만 사용한다.
# ------------------------------------------------------------
def check_clip_dependency():
    info = {"git_found": shutil.which("git") is not None, "clip_import": False, "error": None}
    try:
        import clip  # ultralytics/CLIP 설치 후 보통 이 이름으로 import됨
        info["clip_import"] = True
    except Exception as e:
        info["error"] = repr(e)
    return info

CLIP_STATUS = check_clip_dependency()
print("CLIP_STATUS:", CLIP_STATUS)
if not CLIP_STATUS["git_found"] or not CLIP_STATUS["clip_import"]:
    print("YOLO-World set_classes가 실패할 수 있습니다.")
    print("권장 설치 명령: python -m pip install --no-cache-dir \"git+https://github.com/ultralytics/CLIP.git\"")
    print("Git이 없으면 먼저 Git for Windows 설치 후 새 CMD/Jupyter에서 다시 실행하세요.")

# 기본 YOLO
base_yolo = YOLO(BASE_YOLO_MODEL_PATH)
try:
    base_yolo.to(str(DEVICE))
except Exception as e:
    print("base_yolo.to failed, continue:", repr(e))
print("base_yolo loaded:", BASE_YOLO_MODEL_PATH)

# pose 모델: yolo11n-pose 우선, 실패하면 yolov8n-pose
pose_model = None
pose_model_name = None
for cand in POSE_MODEL_CANDIDATES:
    try:
        pose_model = YOLO(cand)
        try:
            pose_model.to(str(DEVICE))
        except Exception:
            pass
        pose_model_name = cand
        print("pose_model loaded:", cand)
        break
    except Exception as e:
        print("pose_model load failed:", cand, repr(e))

# YOLO-World: set_classes 성공 여부를 반드시 확인한다.
world_model = None
world_model_name = None
world_model_active = False
world_set_classes_error = None
world_classes = [
    "person", "bicycle", "motorcycle", "motorbike", "scooter", "electric scooter",
    "kickboard", "moped", "helmet", "no helmet", "road", "sidewalk", "crosswalk"
]
for cand in YOLO_WORLD_MODEL_CANDIDATES:
    try:
        wm = YOLO(cand)
        try:
            wm.to(str(DEVICE))
        except Exception:
            pass
        wm.set_classes(world_classes)
        world_model = wm
        world_model_name = cand
        world_model_active = True
        print("world_model loaded and set_classes OK:", cand)
        break
    except Exception as e:
        world_set_classes_error = repr(e)
        print("world_model load/set_classes failed:", cand, world_set_classes_error)
        world_model = None
        world_model_name = None
        world_model_active = False

# SegFormer: 선택. 실패해도 주행은 bbox 기반 후보로 계속 진행.
seg_processor = None
seg_model = None
seg_device = None
seg_model_id = None
try:
    from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
    if torch is not None:
        seg_device = DEVICE
        seg_model_id = SEG_MODEL_ID_GPU if str(DEVICE) == "cuda" else SEG_MODEL_ID_CPU
        seg_processor = SegformerImageProcessor.from_pretrained(seg_model_id)
        seg_model = SegformerForSemanticSegmentation.from_pretrained(seg_model_id).to(seg_device)
        seg_model.eval()
        print("SegFormer loaded:", seg_model_id, "device:", seg_device)
except Exception as e:
    print("SegFormer load failed. sidewalk segmentation fallback will be limited:", repr(e))
    seg_processor = None
    seg_model = None


# ------------------------------------------------------------
# 헬멧 모델 자동 탐색
# HELMET_MODEL_PATH가 None이면 이전 cf6/tog/aio 학습 결과 위치에서 best.pt를 찾아본다.
# 단, 실제 모델이 없으면 helmet_backend="none"이며 헬멧 판정은 적용되지 않는다.
# ------------------------------------------------------------
def discover_helmet_model_path():
    candidates = []
    roots = [AIO_DIR, OUTPUT_ROOT / "aio", OUTPUT_ROOT / "prediction", OUTPUT_ROOT, Path.cwd()]
    explicit = [
        AIO_DIR / "runs_helmet" / "bike_helmet_yolov8" / "weights" / "best.pt",
        OUTPUT_ROOT / "aio" / "runs_helmet" / "bike_helmet_yolov8" / "weights" / "best.pt",
        OUTPUT_ROOT / "runs_cf6_helmet" / "bike_helmet_yolov8" / "weights" / "best.pt",
        OUTPUT_ROOT / "runs_helmet" / "bike_helmet_yolov8" / "weights" / "best.pt",
    ]
    for p in explicit:
        candidates.append(Path(p))
    for root in roots:
        try:
            if Path(root).exists():
                candidates.extend(Path(root).rglob("best.pt"))
        except Exception:
            pass
    # helmet, bike, cf6, roboflow, runs가 들어간 경로 우선
    scored = []
    for p in candidates:
        ps = str(p).lower()
        if not p.exists():
            continue
        score = 0
        for kw in ["helmet", "helm", "bike_helmet", "cf6", "roboflow", "runs_helmet", "runs_cf6"]:
            if kw in ps:
                score += 2
        if "weights" in ps and p.name == "best.pt":
            score += 1
        scored.append((score, p.stat().st_mtime, p))
    if not scored:
        return None, []
    scored.sort(reverse=True)
    return scored[0][2], [str(x[2]) for x in scored[:10]]

if (not HELMET_MODEL_PATH) and AUTO_DISCOVER_HELMET_MODEL:
    _auto_helmet_path, _helmet_candidates = discover_helmet_model_path()
    if _auto_helmet_path is not None:
        HELMET_MODEL_PATH = str(_auto_helmet_path)
        print("auto helmet model found:", HELMET_MODEL_PATH)
        print("helmet candidates top:", _helmet_candidates[:3])
    else:
        print("auto helmet model not found. HELMET_MODEL_PATH를 직접 best.pt로 지정해야 헬멧 판정이 적용됩니다.")

# 헬멧 모델: local YOLO 또는 Roboflow hosted 중 하나
helmet_model = None
helmet_backend = "none"
rf_client = None
rf_hosted_ready = False

if HELMET_MODEL_PATH:
    try:
        helmet_model = YOLO(HELMET_MODEL_PATH)
        try:
            helmet_model.to(str(DEVICE))
        except Exception:
            pass
        helmet_backend = "local_yolo"
        print("helmet_model loaded:", HELMET_MODEL_PATH)
        try:
            print("helmet_model names:", helmet_model.names)
        except Exception:
            pass
    except Exception as e:
        print("helmet_model load failed:", repr(e))

if USE_ROBOFLOW_HOSTED_INFERENCE and helmet_backend != "local_yolo":
    if not ROBOFLOW_API_KEY:
        print("ROBOFLOW_API_KEY가 없어 hosted inference를 사용할 수 없습니다.")
    else:
        try:
            from inference_sdk import InferenceHTTPClient
            rf_client = InferenceHTTPClient(api_url="https://detect.roboflow.com", api_key=ROBOFLOW_API_KEY)
            rf_hosted_ready = True
            helmet_backend = "roboflow_hosted"
            print("Roboflow hosted inference ready:", ROBOFLOW_MODEL_ID)
            print("helmet hosted call limit:", MAX_HELMET_CHECKS_PER_VIDEO, "per video / stride:", HELMET_CHECK_SAMPLE_STRIDE)
        except Exception as e:
            print("Roboflow hosted inference init failed:", repr(e))
elif USE_ROBOFLOW_HOSTED_INFERENCE and helmet_backend == "local_yolo":
    print("Roboflow API key is present, but local helmet YOLO is available. Using local_yolo for speed.")

MODEL_STATUS = {
    "aio_version": AIO_VERSION,
    "device": str(DEVICE),
    "base_yolo": BASE_YOLO_MODEL_PATH,
    "pose_model": pose_model_name,
    "world_model": world_model_name,
    "world_model_active": world_model_active,
    "world_classes": world_classes if world_model_active else [],
    "world_set_classes_error": world_set_classes_error,
    "clip_status": CLIP_STATUS,
    "segformer": seg_model_id,
    "tracker": TRACKER_YAML if USE_YOLO_TRACKER else "custom_greedy",
    "lap_status": LAP_STATUS,
    "helmet_backend": helmet_backend,
    "helmet_model": HELMET_MODEL_PATH if helmet_backend == "local_yolo" else None,
    "roboflow_model_id": ROBOFLOW_MODEL_ID if helmet_backend == "roboflow_hosted" else None,
    "helmet_limits": {
        "helmet_enable": HELMET_ENABLE,
        "helmet_check_sample_stride": HELMET_CHECK_SAMPLE_STRIDE,
        "max_helmet_checks_per_video": MAX_HELMET_CHECKS_PER_VIDEO,
        "max_helmet_pairs_per_frame": MAX_HELMET_PAIRS_PER_FRAME,
        "early_stop_no_helmet_hits": HELMET_EARLY_STOP_NO_HELMET_HITS,
        "early_stop_confident_hits": HELMET_EARLY_STOP_CONFIDENT_HITS,
    },
    "primary_label_min_score": PRIMARY_LABEL_MIN_SCORE,
    "primary_label_margin": PRIMARY_LABEL_MARGIN,
}
print("MODEL_STATUS:", json.dumps(MODEL_STATUS, ensure_ascii=False, indent=2))


cv2 OK
numpy OK
tqdm OK
torch OK, cuda: True device: cuda
gpu: NVIDIA GeForce RTX 3060
ultralytics YOLO OK
PIL OK
lap OK
CLIP_STATUS: {'git_found': True, 'clip_import': True, 'error': None}
base_yolo loaded: yolov8n.pt
pose_model loaded: yolo11n-pose.pt
world_model loaded and set_classes OK: yolov8s-world.pt


Loading weights:   0%|          | 0/1172 [00:00<?, ?it/s]

SegFormer loaded: nvidia/segformer-b5-finetuned-cityscapes-1024-1024 device: cuda
auto helmet model not found. HELMET_MODEL_PATH를 직접 best.pt로 지정해야 헬멧 판정이 적용됩니다.
MODEL_STATUS: {
  "aio_version": "qq_fast_helmet_primary_label_v1",
  "device": "cuda",
  "base_yolo": "yolov8n.pt",
  "pose_model": "yolo11n-pose.pt",
  "world_model": "yolov8s-world.pt",
  "world_model_active": true,
  "world_classes": [
    "person",
    "bicycle",
    "motorcycle",
    "motorbike",
    "scooter",
    "electric scooter",
    "kickboard",
    "moped",
    "helmet",
    "no helmet",
    "road",
    "sidewalk",
    "crosswalk"
  ],
  "world_set_classes_error": null,
  "clip_status": {
    "git_found": true,
    "clip_import": true,
    "error": null
  },
  "segformer": "nvidia/segformer-b5-finetuned-cityscapes-1024-1024",
  "tracker": "bytetrack.yaml",
  "lap_status": {
    "lap_import": true,
    "error": null
  },
  "helmet_backend": "none",
  "helmet_model": null,
  "roboflow_model_id": null,
  "helmet_limit

In [3]:

# ============================================================
# Cell 3. 영상 탐색
# ============================================================
video_paths = sorted([p for p in VIDEO_ROOT.rglob("*") if p.is_file() and p.suffix in VIDEO_EXTS])
if MAX_VIDEOS is not None:
    video_paths = video_paths[:MAX_VIDEOS]

print("찾은 영상 수:", len(video_paths))
for p in video_paths[:30]:
    print(p)

if not video_paths:
    raise RuntimeError("원천데이터 안에서 영상 파일을 찾지 못했습니다.")


찾은 영상 수: 22
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 구역 내 지속 배회\lo_e1281\lo_e1281_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 구역 내 지속 배회\lo_e1281\lo_e1281_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 인물을 뒤따라가며 배회\fp_e1520\fp_e1520_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 인물을 뒤따라가며 배회\fp_e1520\fp_e1520_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\싸움\신체적 충돌을 동반한 싸움\ph_e2063\ph_e2063_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\싸움\신체적 충돌을 동반한 싸움\ph_e2063\ph_e2063_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\오토바이 인도 주행\ms_e3500\ms_e3500_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\오토바이 인도 주행\ms_e3500\ms_e3500_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\자전거 인도 주행\bs_e4387\bs_e4387_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\자전거 인도 주행\bs_e4387\bs_e4387_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\전동킥보드 인도 주행\es_e3768\es_e3768_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데

In [4]:
# ============================================================
# Cell 3B. 실행 전 점검
# ============================================================
print("="*80)
print("TOG preflight check")
print("VIDEO_ROOT:", VIDEO_ROOT)
print("video count:", len(video_paths))
print("AIO_FEATURE_DIR:", AIO_FEATURE_DIR)
print("OVERWRITE_FEATURES:", OVERWRITE_FEATURES)
print("USE_YOLO_TRACKER:", USE_YOLO_TRACKER, "TRACKER_YAML:", TRACKER_YAML)
print("world_model_active:", MODEL_STATUS.get("world_model_active"))
print("helmet_backend:", MODEL_STATUS.get("helmet_backend"))
print("helmet_model:", MODEL_STATUS.get("helmet_model"))
print("helmet_limits:", MODEL_STATUS.get("helmet_limits"))
print("primary_label_min_score:", MODEL_STATUS.get("primary_label_min_score"))
print("segformer:", MODEL_STATUS.get("segformer"))
print("lap_status:", MODEL_STATUS.get("lap_status"))

if len(video_paths) == 0:
    raise RuntimeError("영상이 0개입니다. VIDEO_ROOT를 확인하세요.")
if USE_YOLO_TRACKER and not MODEL_STATUS.get("lap_status", {}).get("lap_import", False):
    print("주의: lap이 현재 커널에 import되지 않습니다. ByteTrack이 실패하면 자동 detect fallback으로 진행합니다.")
if not MODEL_STATUS.get("world_model_active"):
    print("주의: YOLO-World가 비활성입니다. scooter/kickboard 보강이 약해집니다. Git + ultralytics/CLIP 설치를 확인하세요.")
if MODEL_STATUS.get("helmet_backend") == "none":
    print("주의: 헬멧 모델이 연결되지 않았습니다. ROBOFLOW_API_KEY 환경변수를 넣거나 HELMET_MODEL_PATH에 best.pt를 넣어야 헬멧 판정이 됩니다.")
print("="*80)


TOG preflight check
VIDEO_ROOT: C:\Users\우성한\Sample (1)\Sample\01.원천데이터
video count: 22
AIO_FEATURE_DIR: C:\Users\우성한\outputs\prediction\qq\aio_features
OVERWRITE_FEATURES: True
USE_YOLO_TRACKER: True TRACKER_YAML: bytetrack.yaml
world_model_active: True
helmet_backend: none
helmet_model: None
helmet_limits: {'helmet_enable': True, 'helmet_check_sample_stride': 3, 'max_helmet_checks_per_video': 16, 'max_helmet_pairs_per_frame': 1, 'early_stop_no_helmet_hits': 2, 'early_stop_confident_hits': 5}
primary_label_min_score: 0.3
segformer: nvidia/segformer-b5-finetuned-cityscapes-1024-1024
lap_status: {'lap_import': True, 'error': None}
주의: 헬멧 모델이 연결되지 않았습니다. ROBOFLOW_API_KEY 환경변수를 넣거나 HELMET_MODEL_PATH에 best.pt를 넣어야 헬멧 판정이 됩니다.


In [5]:
# ============================================================
# Cell 4. 공통 유틸 함수  [track_id/helmet crop 보강]
# ============================================================
def safe_stem_from_path(path: Path, root: Path):
    rel = path.relative_to(root)
    return "__".join(rel.with_suffix("").parts)


def clamp(v, lo, hi):
    return max(lo, min(hi, v))


def bbox_center(b):
    x1, y1, x2, y2 = b
    return ((x1 + x2) / 2, (y1 + y2) / 2)


def bbox_area(b):
    x1, y1, x2, y2 = b
    return max(0, x2 - x1) * max(0, y2 - y1)


def bbox_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    union = bbox_area(a) + bbox_area(b) - inter
    return inter / union if union > 0 else 0


def bbox_overlap_ratio(a, b):
    # a 기준으로 b와 겹치는 비율
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area = bbox_area(a)
    return inter / area if area > 0 else 0


def dist(p1, p2):
    return math.sqrt((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2)


def vec(a, b):
    return (b[0] - a[0], b[1] - a[1])


def norm(v):
    return math.sqrt(v[0] ** 2 + v[1] ** 2)


def cosine(v1, v2):
    n1, n2 = norm(v1), norm(v2)
    if n1 <= 1e-6 or n2 <= 1e-6:
        return 0
    return (v1[0] * v2[0] + v1[1] * v2[1]) / (n1 * n2)


def video_meta(video_path):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return {"fps": None, "width": None, "height": None, "frame_count": None}
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return {
        "fps": float(fps) if fps and fps > 0 else None,
        "width": width,
        "height": height,
        "frame_count": frame_count,
    }


def yolo_objects(result, names, min_conf=0.0):
    objs = []
    if result is None or result.boxes is None:
        return objs
    track_ids = None
    try:
        if getattr(result.boxes, "id", None) is not None:
            track_ids = result.boxes.id.detach().cpu().numpy().astype(int).tolist()
    except Exception:
        track_ids = None
    for idx, box in enumerate(result.boxes):
        cls_id = int(box.cls[0].item() if hasattr(box.cls[0], "item") else box.cls[0])
        cls_name = names.get(cls_id, str(cls_id)) if isinstance(names, dict) else str(cls_id)
        conf = float(box.conf[0].item() if hasattr(box.conf[0], "item") else box.conf[0])
        if conf < min_conf:
            continue
        bbox = [round(float(v), 2) for v in box.xyxy[0].tolist()]
        item = {
            "class": cls_name,
            "confidence": round(conf, 4),
            "bbox": bbox,
            "center": [round(v, 2) for v in bbox_center(bbox)],
            "area": round(bbox_area(bbox), 2),
        }
        if track_ids is not None and idx < len(track_ids):
            item["track_id"] = int(track_ids[idx])
        objs.append(item)
    return objs


def reset_yolo_tracker(model):
    """
    영상마다 ByteTrack 상태를 초기화한다.
    중요: model.predictor.trackers = None 으로 직접 건드리면
    Ultralytics 내부에서 trackers[0] 접근 시 NoneType 오류가 날 수 있다.
    predictor 전체를 None으로 두면 다음 호출에서 안전하게 재생성된다.
    """
    try:
        model.predictor = None
    except Exception:
        pass


def run_base_yolo_or_tracker(frame):
    """
    1순위: YOLO ByteTrack.
    실패하면 predictor를 초기화하고 일반 YOLO detect로 fallback한다.
    그래서 tracker/lap 문제 때문에 전체 feature 생성이 중단되지 않는다.
    """
    if USE_YOLO_TRACKER:
        try:
            outs = base_yolo.track(frame, persist=True, tracker=TRACKER_YAML, verbose=False)
            if outs is None or len(outs) == 0 or outs[0] is None:
                raise RuntimeError("base_yolo.track returned empty result")
            return outs[0], "yolo_track"
        except Exception as e:
            tracker_error = repr(e)[:180]
            try:
                base_yolo.predictor = None
            except Exception:
                pass
            try:
                outs = base_yolo(frame, verbose=False)
                if outs is None or len(outs) == 0 or outs[0] is None:
                    raise RuntimeError("base_yolo detect returned empty result")
                return outs[0], f"detect_fallback_after_tracker_error:{tracker_error}"
            except Exception as e2:
                raise RuntimeError(
                    "base_yolo track and detect both failed. "
                    f"track_error={repr(e)}, detect_error={repr(e2)}"
                )

    outs = base_yolo(frame, verbose=False)
    if outs is None or len(outs) == 0 or outs[0] is None:
        raise RuntimeError("base_yolo detect returned empty result")
    return outs[0], "yolo_detect"


def extract_pose_people(result):
    people = []
    if result is None or result.boxes is None:
        return people
    boxes = result.boxes
    kpts = getattr(result, "keypoints", None)
    xy_data = None
    conf_data = None
    if kpts is not None:
        try:
            xy_data = kpts.xy.cpu().numpy()
            conf_data = kpts.conf.cpu().numpy() if kpts.conf is not None else None
        except Exception:
            xy_data = None
            conf_data = None
    for i, box in enumerate(boxes):
        try:
            conf = float(box.conf[0])
            bbox = [round(float(v), 2) for v in box.xyxy[0].tolist()]
        except Exception:
            continue
        item = {"bbox": bbox, "confidence": round(conf, 4), "center": [round(v,2) for v in bbox_center(bbox)]}
        if xy_data is not None and i < len(xy_data):
            pts = xy_data[i].tolist()
            if conf_data is not None and i < len(conf_data):
                pc = conf_data[i].tolist()
            else:
                pc = [None] * len(pts)
            item["keypoints"] = [
                {"x": round(float(p[0]), 2), "y": round(float(p[1]), 2), "conf": None if c is None else round(float(c), 4)}
                for p, c in zip(pts, pc)
            ]
        people.append(item)
    return people


def compute_pose_agitation(pose_people):
    vals = []
    for p in pose_people:
        kps = p.get("keypoints", [])
        good = [(k["x"], k["y"]) for k in kps if k.get("conf") is None or k.get("conf", 0) >= POSE_MIN_CONF]
        if len(good) < 5:
            continue
        xs = [x for x, y in good]
        ys = [y for x, y in good]
        spread = (max(xs) - min(xs)) + (max(ys) - min(ys))
        area = max(1.0, math.sqrt(bbox_area(p["bbox"])))
        vals.append(spread / area)
    return float(np.mean(vals)) if vals else 0.0


def optical_flow_stats(prev_gray, gray):
    if prev_gray is None or gray is None:
        return {"mean_mag": 0.0, "p90_mag": 0.0, "strong_ratio": 0.0}
    h, w = gray.shape[:2]
    if w > FLOW_RESIZE_WIDTH:
        scale = FLOW_RESIZE_WIDTH / w
        gray_s = cv2.resize(gray, (FLOW_RESIZE_WIDTH, int(h * scale)))
        prev_s = cv2.resize(prev_gray, (FLOW_RESIZE_WIDTH, int(h * scale)))
    else:
        gray_s, prev_s = gray, prev_gray
    flow = cv2.calcOpticalFlowFarneback(prev_s, gray_s, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    mag, ang = cv2.cartToPolar(flow[...,0], flow[...,1])
    return {
        "mean_mag": round(float(np.mean(mag)), 4),
        "p90_mag": round(float(np.percentile(mag, 90)), 4),
        "strong_ratio": round(float(np.mean(mag >= FLOW_MAG_STRONG)), 4),
    }


def get_seg_mask(frame_bgr):
    if seg_model is None or seg_processor is None:
        return None
    try:
        rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        inputs = seg_processor(images=rgb, return_tensors="pt")
        inputs = {k: v.to(seg_device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = seg_model(**inputs)
        logits = outputs.logits
        up = torch.nn.functional.interpolate(
            logits, size=rgb.shape[:2], mode="bilinear", align_corners=False
        )
        pred = up.argmax(dim=1)[0].detach().cpu().numpy().astype(np.uint8)
        return pred
    except RuntimeError as e:
        if "out of memory" in str(e).lower() and torch is not None and torch.cuda.is_available():
            torch.cuda.empty_cache()
        raise


def patch_label_ratio(mask, x, y, radius=18):
    if mask is None:
        return {"sidewalk_ratio": None, "road_ratio": None}
    h, w = mask.shape[:2]
    x = int(clamp(round(x), 0, w - 1))
    y = int(clamp(round(y), 0, h - 1))
    x1, x2 = clamp(x - radius, 0, w - 1), clamp(x + radius, 0, w - 1)
    y1, y2 = clamp(y - radius, 0, h - 1), clamp(y + radius, 0, h - 1)
    patch = mask[y1:y2+1, x1:x2+1]
    if patch.size == 0:
        return {"sidewalk_ratio": 0.0, "road_ratio": 0.0}
    return {
        "sidewalk_ratio": round(float(np.mean(patch == SIDEWALK_ID)), 4),
        "road_ratio": round(float(np.mean(patch == ROAD_ID)), 4),
    }


def bbox_bottom_point(bbox):
    x1, y1, x2, y2 = bbox
    x = (x1 + x2) / 2
    y = y1 + (y2 - y1) * BOTTOM_POINT_Y_RATIO
    return x, y


def bbox_bottom_band_label_ratio(mask, bbox):
    if mask is None:
        return {"sidewalk_band_ratio": None, "road_band_ratio": None}
    h, w = mask.shape[:2]
    x1, y1, x2, y2 = bbox
    bw = x2 - x1
    bh = y2 - y1
    xx1 = int(clamp(x1 + bw * BOTTOM_BAND_X_MARGIN, 0, w - 1))
    xx2 = int(clamp(x2 - bw * BOTTOM_BAND_X_MARGIN, 0, w - 1))
    yy1 = int(clamp(y1 + bh * BOTTOM_BAND_Y1, 0, h - 1))
    yy2 = int(clamp(y1 + bh * BOTTOM_BAND_Y2, 0, h - 1))
    if xx1 >= xx2 or yy1 >= yy2:
        return {"sidewalk_band_ratio": 0.0, "road_band_ratio": 0.0}
    patch = mask[yy1:yy2+1, xx1:xx2+1]
    return {
        "sidewalk_band_ratio": round(float(np.mean(patch == SIDEWALK_ID)), 4),
        "road_band_ratio": round(float(np.mean(patch == ROAD_ID)), 4),
    }


# ------------------------------------------------------------
# 헬멧 모델/데이터셋 선택 함수
# ------------------------------------------------------------
def download_roboflow_helmet_dataset():
    if not ROBOFLOW_API_KEY:
        raise RuntimeError("ROBOFLOW_API_KEY가 없습니다. CMD에서 set ROBOFLOW_API_KEY=... 후 Jupyter를 재시작하세요.")
    try:
        from roboflow import Roboflow
    except Exception as e:
        raise RuntimeError("roboflow 패키지가 없습니다. python -m pip install roboflow 후 실행하세요.") from e
    RF_DATASET_DIR.mkdir(parents=True, exist_ok=True)
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
    version = project.version(RF_VERSION)
    dataset = version.download(RF_EXPORT_FORMAT, location=str(RF_DATASET_DIR / f"{RF_PROJECT}-{RF_VERSION}"))
    data_yaml = Path(dataset.location) / "data.yaml"
    print("dataset location:", dataset.location)
    print("data.yaml exists:", data_yaml.exists(), data_yaml)
    return data_yaml


def train_helmet_yolov8(data_yaml, epochs=30, imgsz=640, batch=8, model="yolov8n.pt"):
    data_yaml = Path(data_yaml)
    if not data_yaml.exists():
        raise FileNotFoundError(data_yaml)
    model_obj = YOLO(model)
    result = model_obj.train(
        data=str(data_yaml),
        epochs=epochs,
        imgsz=imgsz,
        batch=batch,
        project=str(RF_TRAIN_PROJECT_DIR),
        name="bike_helmet_yolov8",
        device=0 if str(DEVICE) == "cuda" else "cpu",
    )
    best_pt = RF_TRAIN_PROJECT_DIR / "bike_helmet_yolov8" / "weights" / "best.pt"
    print("training done. best.pt:", best_pt, "exists:", best_pt.exists())
    return best_pt

# 필요할 때만 별도 실행:
# RF_DATA_YAML = download_roboflow_helmet_dataset()
# HELMET_MODEL_PATH = str(train_helmet_yolov8(RF_DATA_YAML, epochs=30, imgsz=640, batch=8))
# 이후 Cell 1의 HELMET_MODEL_PATH에 best.pt 경로를 넣고 Cell 2부터 다시 실행.

saved_crop_count = 0

def crop_head_and_upper_body(image_rgb: np.ndarray, person_bbox, mode="head"):
    if image_rgb is None:
        return None
    h_img, w_img = image_rgb.shape[:2]
    x1, y1, x2, y2 = [int(round(v)) for v in person_bbox]
    bw = x2 - x1
    bh = y2 - y1
    if bw <= 0 or bh <= 0:
        return None
    if mode == "head":
        cy2 = y1 + int(bh * 0.45)
        mx = int(bw * 0.35)
        my = int(bh * 0.12)
    else:
        cy2 = y1 + int(bh * 0.62)
        mx = int(bw * 0.25)
        my = int(bh * 0.08)
    cx1 = max(0, x1 - mx)
    cy1 = max(0, y1 - my)
    cx2 = min(w_img, x2 + mx)
    cy2 = min(h_img, cy2 + my)
    if cx1 >= cx2 or cy1 >= cy2:
        return None
    return image_rgb[cy1:cy2, cx1:cx2]


def normalize_helmet_label(label: str):
    label = str(label).lower().strip().replace("-", "_").replace(" ", "_")
    no_helmet_labels = {
        "no_helmet", "nohelmet", "without_helmet", "withouthelmet", "bare_head", "barehead",
        "rider_without_helmet", "without", "not_helmet"
    }
    helmet_labels = {"helmet", "with_helmet", "withhelmet", "hardhat", "hard_hat", "rider_with_helmet", "with"}
    if label in no_helmet_labels or "without" in label or "no_helmet" in label or "nohelmet" in label:
        return "no_helmet"
    if label in helmet_labels or "with_helmet" in label or "withhelmet" in label or label == "helmet":
        return "helmet"
    return label


def save_debug_crop(crop, video_name, frame_id, pair_idx, mode):
    global saved_crop_count
    if not SAVE_HELMET_CROPS or crop is None or saved_crop_count >= MAX_SAVED_HELMET_CROPS:
        return None
    stem = Path(video_name).stem
    path = HELMET_CROP_DIR / f"{stem}_f{frame_id}_p{pair_idx}_{mode}.jpg"
    cv2.imwrite(str(path), cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))
    saved_crop_count += 1
    return str(path)


def predict_helmet_local_yolo(crop):
    try:
        pred = helmet_model.predict(crop, verbose=False, conf=HELMET_CONF, device=0 if str(DEVICE) == "cuda" else "cpu")
    except TypeError:
        pred = helmet_model.predict(crop, verbose=False, conf=HELMET_CONF)
    detections = []
    for res in pred:
        names = getattr(res, "names", getattr(helmet_model, "names", {}))
        boxes = getattr(res, "boxes", None)
        if boxes is None:
            continue
        for b in boxes:
            cls_id = int(b.cls[0].item())
            conf = float(b.conf[0].item())
            label = names.get(cls_id, str(cls_id)) if isinstance(names, dict) else str(cls_id)
            norm_label = normalize_helmet_label(label)
            xyxy = [int(round(v)) for v in b.xyxy[0].detach().cpu().tolist()]
            detections.append({"label": label, "normalized": norm_label, "confidence": round(conf, 4), "bbox_crop": xyxy})
    return detections


def predict_helmet_roboflow(crop):
    with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp:
        tmp_path = tmp.name
    try:
        cv2.imwrite(tmp_path, cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))
        response = rf_client.infer(tmp_path, model_id=ROBOFLOW_MODEL_ID)
    finally:
        try:
            os.remove(tmp_path)
        except Exception:
            pass
    detections = []
    for p in response.get("predictions", []):
        label = p.get("class", "")
        conf = float(p.get("confidence", 0.0))
        norm_label = normalize_helmet_label(label)
        x = float(p.get("x", 0)); y = float(p.get("y", 0)); w = float(p.get("width", 0)); h = float(p.get("height", 0))
        xyxy = [int(x - w/2), int(y - h/2), int(x + w/2), int(y + h/2)]
        detections.append({"label": label, "normalized": norm_label, "confidence": round(conf, 4), "bbox_crop": xyxy})
    return detections


def decide_helmet_from_detections(detections):
    if not detections:
        return "unknown", "helmet model returned no detection", []
    recognized = [d for d in detections if d.get("normalized") in {"helmet", "no_helmet"}]
    if not recognized:
        best = max(detections, key=lambda x: x["confidence"])
        return "unknown", f"helmet model labels not recognized: {best['label']}", detections
    no_hits = [d for d in recognized if d["normalized"] == "no_helmet"]
    hel_hits = [d for d in recognized if d["normalized"] == "helmet"]
    best_no = max(no_hits, key=lambda x: x["confidence"], default=None)
    best_hel = max(hel_hits, key=lambda x: x["confidence"], default=None)
    if best_no and (best_hel is None or best_no["confidence"] >= best_hel["confidence"] - 0.05):
        return "no_helmet", f"detected no_helmet conf={best_no['confidence']}", detections
    if best_hel:
        return "helmet", f"detected helmet conf={best_hel['confidence']}", detections
    return "unknown", "no recognized helmet/no_helmet detection", detections


def judge_helmet(image_rgb, person_bbox, video_name=None, frame_id=None, pair_idx=0):
    if helmet_backend == "none":
        return "unknown", "helmet model is not connected. Set HELMET_MODEL_PATH or Roboflow API.", []
    all_detections = []
    crop_paths = []
    for mode in ["head", "upper"]:
        crop = crop_head_and_upper_body(image_rgb, person_bbox, mode=mode)
        if crop is None:
            continue
        crop_path = save_debug_crop(crop, video_name or "video", frame_id or -1, pair_idx, mode)
        if crop_path:
            crop_paths.append(crop_path)
        try:
            if helmet_backend == "local_yolo":
                dets = predict_helmet_local_yolo(crop)
            elif helmet_backend == "roboflow_hosted":
                dets = predict_helmet_roboflow(crop)
            else:
                dets = []
        except Exception as e:
            return "unknown", f"helmet predict failed on {mode}: {repr(e)}", []
        for d in dets:
            d["crop_mode"] = mode
        all_detections.extend(dets)
        status, reason, _ = decide_helmet_from_detections(dets)
        if status in {"helmet", "no_helmet"}:
            if crop_paths:
                reason += f"; crop_saved={crop_paths[-1]}"
            return status, reason, all_detections
    status, reason, detections = decide_helmet_from_detections(all_detections)
    if crop_paths:
        reason += f"; crops_saved={len(crop_paths)}"
    return status, reason, detections


## 선택: Roboflow 데이터셋을 로컬 YOLO 헬멧 모델로 학습하기

Hosted inference가 느리면 아래 셀의 주석을 해제해서 한 번만 학습하세요. 학습 후 출력되는 `best.pt` 경로를 Cell 1의 `HELMET_MODEL_PATH`에 넣으면 이후부터는 `helmet_backend: local_yolo`로 빠르게 처리됩니다.


In [6]:
# 선택 실행: 시간이 걸릴 수 있음. 필요할 때만 주석 해제해서 사용하세요.
# RF_DATA_YAML = download_roboflow_helmet_dataset()
# HELMET_MODEL_PATH = str(train_helmet_yolov8(RF_DATA_YAML, epochs=20, imgsz=640, batch=8))
# print('학습된 헬멧 모델:', HELMET_MODEL_PATH)
# print('이 경로를 Cell 1의 HELMET_MODEL_PATH에 넣고 Cell 2부터 다시 실행하세요.')


In [7]:
# ============================================================
# Cell 5. AIO 프레임 특징 JSON 생성  [YOLO tracker + YOLO-World active check + helmet evidence]
# ============================================================
def create_aio_features_for_video(video_path, output_json_path):
    meta = video_meta(video_path)
    fps = meta.get("fps") or 30.0
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"영상 열기 실패: {video_path}")

    reset_yolo_tracker(base_yolo)

    frames = []
    frame_idx = 0
    sampled_idx = 0
    prev_gray = None
    tracker_modes = Counter()
    helmet_checked_count = 0
    helmet_no_hits_runtime = 0
    helmet_confident_hits_runtime = 0
    helmet_skipped_by_limit = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if MAX_FRAMES_PER_VIDEO is not None and frame_idx >= MAX_FRAMES_PER_VIDEO:
            break

        if frame_idx % FRAME_INTERVAL == 0:
            t = frame_idx / fps if fps else sampled_idx
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            flow = optical_flow_stats(prev_gray, gray)
            prev_gray = gray

            # 기본 YOLO + ByteTrack. 실패하면 일반 YOLO detect fallback.
            base_result, tracker_mode = run_base_yolo_or_tracker(frame)
            tracker_modes[tracker_mode] += 1
            base_objects = yolo_objects(base_result, base_yolo.names, min_conf=0.0)

            persons = [o for o in base_objects if o["class"] == "person" and o["confidence"] >= PERSON_MIN_CONF]
            base_tw = [o for o in base_objects if o["class"] in TWO_WHEELER_CLASSES and o["confidence"] >= VEHICLE_MIN_CONF]

            # YOLO-World 객체 보강: set_classes 성공한 경우에만 사용.
            world_objects = []
            if world_model is not None and world_model_active:
                try:
                    wr = world_model(frame, verbose=False, conf=WORLD_MIN_CONF)[0]
                    world_objects = yolo_objects(wr, world_model.names, min_conf=WORLD_MIN_CONF)
                except Exception:
                    world_objects = []

            world_tw = []
            world_helmet_candidates = []
            for o in world_objects:
                cls = str(o.get("class", "")).lower()
                if cls in WORLD_TWO_WHEELER_HINTS or any(h in cls for h in ["scooter", "bike", "motor", "moped", "kickboard"]):
                    world_tw.append(o)
                if "helmet" in cls:
                    world_helmet_candidates.append(o)

            # pose
            pose_people = []
            pose_agitation = 0.0
            if pose_model is not None:
                try:
                    pr = pose_model(frame, verbose=False)[0]
                    pose_people = extract_pose_people(pr)
                    pose_agitation = compute_pose_agitation(pose_people)
                except Exception:
                    pose_people = []
                    pose_agitation = 0.0

            # segmentation
            seg_mask = get_seg_mask(frame)

            two_wheelers = base_tw + world_tw
            sidewalk_evidence = []
            for obj in two_wheelers:
                bx = obj.get("bbox")
                if not bx:
                    continue
                x, y = bbox_bottom_point(bx)
                ratios = patch_label_ratio(seg_mask, x, y, SIDEWALK_PATCH_RADIUS)
                band = bbox_bottom_band_label_ratio(seg_mask, bx)
                sidewalk_evidence.append({
                    "class": obj.get("class"),
                    "confidence": obj.get("confidence"),
                    "bbox": bx,
                    "bottom_point": [round(x,2), round(y,2)],
                    **ratios,
                    **band,
                })

            # rider_like: 사람과 이륜차가 가까운지/겹치는지
            rider_pairs = []
            for p in persons:
                for tw in two_wheelers:
                    d = dist(p["center"], tw["center"])
                    iou = bbox_iou(p["bbox"], tw["bbox"])
                    overlap_person = bbox_overlap_ratio(p["bbox"], tw["bbox"])
                    if d < 260 or iou > 0.01 or overlap_person > 0.015:
                        rider_pairs.append({
                            "person_bbox": p["bbox"],
                            "person_track_id": p.get("track_id"),
                            "two_wheeler_bbox": tw["bbox"],
                            "two_wheeler_class": tw["class"],
                            "two_wheeler_confidence": tw.get("confidence"),
                            "distance": round(d, 2),
                            "iou": round(iou, 4),
                            "person_overlap_ratio": round(overlap_person, 4),
                        })

            # 헬멧 판정: hosted inference는 매우 느릴 수 있으므로 영상당 호출 수를 제한한다.
            helmet_evidence = []
            helmet_skip_reason = None
            should_check_helmet = (
                HELMET_ENABLE
                and rider_pairs
                and helmet_backend != "none"
                and helmet_checked_count < MAX_HELMET_CHECKS_PER_VIDEO
                and (sampled_idx % max(1, HELMET_CHECK_SAMPLE_STRIDE) == 0)
                and helmet_no_hits_runtime < HELMET_EARLY_STOP_NO_HELMET_HITS
                and helmet_confident_hits_runtime < HELMET_EARLY_STOP_CONFIDENT_HITS
            )
            if rider_pairs and helmet_backend == "none":
                helmet_skip_reason = "helmet_backend_none"
            elif rider_pairs and not should_check_helmet:
                if helmet_checked_count >= MAX_HELMET_CHECKS_PER_VIDEO:
                    helmet_skip_reason = "max_helmet_checks_per_video_reached"
                elif sampled_idx % max(1, HELMET_CHECK_SAMPLE_STRIDE) != 0:
                    helmet_skip_reason = "helmet_frame_stride_skip"
                elif helmet_no_hits_runtime >= HELMET_EARLY_STOP_NO_HELMET_HITS:
                    helmet_skip_reason = "early_stop_no_helmet_already_confirmed"
                elif helmet_confident_hits_runtime >= HELMET_EARLY_STOP_CONFIDENT_HITS:
                    helmet_skip_reason = "early_stop_enough_confident_helmet_checks"
                else:
                    helmet_skip_reason = "helmet_check_disabled_or_limited"
                helmet_skipped_by_limit += 1

            if should_check_helmet:
                image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                for pair_idx, pair in enumerate(rider_pairs[:MAX_HELMET_PAIRS_PER_FRAME]):
                    status, reason, dets = judge_helmet(
                        image_rgb,
                        pair["person_bbox"],
                        video_name=video_path.name,
                        frame_id=frame_idx,
                        pair_idx=pair_idx,
                    )
                    helmet_checked_count += 1
                    if status == "no_helmet":
                        helmet_no_hits_runtime += 1
                        helmet_confident_hits_runtime += 1
                    elif status == "helmet":
                        helmet_confident_hits_runtime += 1
                    helmet_evidence.append({
                        "pair_idx": pair_idx,
                        "person_track_id": pair.get("person_track_id"),
                        "helmet_status": status,
                        "reason": reason,
                        "detections": dets,
                    })

            frames.append({
                "frame_id": frame_idx,
                "sampled_id": sampled_idx,
                "time_sec": round(t, 3),
                "tracker_mode": tracker_mode,
                "base_objects": base_objects,
                "world_objects": world_objects,
                "world_model_active": bool(world_model_active),
                "persons": persons,
                "two_wheelers": two_wheelers,
                "rider_pairs": rider_pairs,
                "pose_people": pose_people,
                "pose_agitation": round(float(pose_agitation), 4),
                "flow": flow,
                "sidewalk_evidence": sidewalk_evidence,
                "helmet_evidence": helmet_evidence,
                "helmet_skip_reason": helmet_skip_reason,
                "world_helmet_candidates": world_helmet_candidates,
                "segformer_used": seg_mask is not None,
            })
            sampled_idx += 1

        frame_idx += 1

    cap.release()
    reset_yolo_tracker(base_yolo)

    data = {
        "video_name": video_path.name,
        "video_path": str(video_path),
        "relative_video_path": str(video_path.relative_to(VIDEO_ROOT)),
        "meta": meta,
        "settings": {
            "aio_version": AIO_VERSION,
            "frame_interval": FRAME_INTERVAL,
            "window_seconds": WINDOW_SECONDS,
            "window_stride_seconds": WINDOW_STRIDE_SECONDS,
            "model_status": MODEL_STATUS,
            "tracker_modes": dict(tracker_modes),
            "helmet_checked_count": helmet_checked_count,
            "helmet_no_hits_runtime": helmet_no_hits_runtime,
            "helmet_confident_hits_runtime": helmet_confident_hits_runtime,
            "helmet_skipped_by_limit": helmet_skipped_by_limit,
        },
        "frames": frames,
    }

    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    return data

created = 0
used = 0
failed = []

for idx, video_path in enumerate(tqdm(video_paths, desc="AIO feature JSON"), start=1):
    print(f"\n[{idx}/{len(video_paths)}] 처리 시작: {video_path.name}")
    stem = safe_stem_from_path(video_path, VIDEO_ROOT)
    out_path = AIO_FEATURE_DIR / f"{stem}.json"
    # 코드가 바뀌었기 때문에 기존 JSON을 그대로 쓰고 싶지 않으면 아래 if 블록을 주석 처리하고 재실행.
    if out_path.exists() and not OVERWRITE_FEATURES:
        used += 1
        continue
    try:
        create_aio_features_for_video(video_path, out_path)
        created += 1
    except Exception as e:
        import traceback
        failed.append({
            "video": str(video_path),
            "error": repr(e),
            "traceback": traceback.format_exc(),
        })
        print("\nFAILED:", video_path)
        traceback.print_exc()

print("="*80)
print("AIO feature 생성 완료")
print("전체 영상:", len(video_paths))
print("생성:", created)
print("기존 사용:", used)
print("실패:", len(failed))
if failed:
    for item in failed[:20]:
        print(item)


AIO feature JSON:   0%|                                                                         | 0/22 [00:00<?, ?it/s]


[1/22] 처리 시작: lo_e1281_c1.mp4


AIO feature JSON:   5%|██▉                                                              | 1/22 [00:47<16:34, 47.34s/it]


[2/22] 처리 시작: lo_e1281_c2.mp4


AIO feature JSON:   9%|█████▉                                                           | 2/22 [01:31<15:13, 45.67s/it]

...


[21/22] 처리 시작: ab_e0457_c1.mp4


AIO feature JSON:  95%|█████████████████████████████████████████████████████████████   | 21/22 [16:31<00:50, 50.11s/it]


[22/22] 처리 시작: ab_e0457_c2.mp4


AIO feature JSON: 100%|████████████████████████████████████████████████████████████████| 22/22 [17:20<00:00, 47.30s/it]

AIO feature 생성 완료
전체 영상: 22
생성: 22
기존 사용: 0
실패: 0


In [8]:
# ============================================================
# Cell 6. 트랙 생성, 스토킹 점수, 싸움 점수, 주행/헬멧 점수 계산 함수
# ============================================================
def assign_tracks(frames):
    # 1순위: feature 생성 단계에서 YOLO ByteTrack track_id가 있으면 그대로 사용.
    tracked = defaultdict(list)
    tracked_obs_count = 0
    total_person_count = 0
    for fr in frames:
        for p in fr.get("persons", []):
            total_person_count += 1
            tid = p.get("track_id")
            if tid is None:
                continue
            item = dict(p)
            item["track_id"] = int(tid)
            item["time_sec"] = fr.get("time_sec")
            item["frame_id"] = fr.get("frame_id")
            item["track_source"] = "yolo_tracker"
            tracked[int(tid)].append(item)
            tracked_obs_count += 1
    if total_person_count > 0 and tracked_obs_count / max(1, total_person_count) >= 0.45:
        return dict(tracked)

    # fallback: missed frame을 조금 허용하는 greedy center tracker.
    next_id = 1
    active = {}  # tid -> {center, missed}
    track_history = defaultdict(list)

    for fr in frames:
        persons = sorted(fr.get("persons", []), key=lambda x: -x.get("confidence", 0))
        assigned_tids = set()
        current_updates = {}

        for p in persons:
            c = tuple(p.get("center", bbox_center(p["bbox"])))
            best_tid, best_cost = None, None
            for tid, st in active.items():
                if tid in assigned_tids:
                    continue
                last_c = st["center"]
                d = dist(c, last_c)
                if d > TRACK_MAX_DIST:
                    continue
                size = math.sqrt(max(1.0, bbox_area(p["bbox"])))
                cost = d / max(30.0, size)
                if best_cost is None or cost < best_cost:
                    best_tid, best_cost = tid, cost
            if best_tid is None:
                tid = next_id
                next_id += 1
            else:
                tid = best_tid
            assigned_tids.add(tid)
            current_updates[tid] = {"center": c, "missed": 0}
            item = dict(p)
            item["track_id"] = tid
            item["time_sec"] = fr.get("time_sec")
            item["frame_id"] = fr.get("frame_id")
            item["track_source"] = "custom_greedy"
            track_history[tid].append(item)

        # 미검출 track은 TRACK_MAX_MISSED 프레임까지 유지
        for tid, st in active.items():
            if tid not in current_updates:
                missed = st.get("missed", 0) + 1
                if missed <= TRACK_MAX_MISSED:
                    current_updates[tid] = {"center": st["center"], "missed": missed}
        active = current_updates

    return dict(track_history)


def track_motion(track):
    pts = [tuple(x["center"]) for x in track]
    if len(pts) < 2:
        return {"length": len(pts), "distance": 0, "displacement": 0, "direction_changes": 0, "mean_speed": 0}
    segs = [dist(pts[i-1], pts[i]) for i in range(1, len(pts))]
    displacement = dist(pts[0], pts[-1])
    direction_changes = 0
    for i in range(2, len(pts)):
        v1 = vec(pts[i-2], pts[i-1])
        v2 = vec(pts[i-1], pts[i])
        if cosine(v1, v2) < 0.2 and norm(v1) > 10 and norm(v2) > 10:
            direction_changes += 1
    total_time = max(1e-6, track[-1]["time_sec"] - track[0]["time_sec"])
    return {
        "length": len(pts),
        "distance": round(sum(segs), 3),
        "displacement": round(displacement, 3),
        "direction_changes": direction_changes,
        "mean_speed": round(sum(segs) / total_time, 3),
        "source": track[0].get("track_source", "unknown"),
    }


def compute_stalking_score(frames, tracks):
    track_ids = [tid for tid, tr in tracks.items() if len(tr) >= MIN_TRACK_LEN_FOR_STALKING]
    follow_events = 0
    total_pair_events = 0
    stable_events = 0
    long_track_bonus = 0
    best_pair = None
    best_pair_score = 0

    for tid in track_ids:
        m = track_motion(tracks[tid])
        if m["length"] >= MIN_TRACK_LEN_FOR_STALKING and m["displacement"] > 80:
            long_track_bonus += 1

    by_time = defaultdict(dict)
    for tid, tr in tracks.items():
        for obs in tr:
            by_time[obs["time_sec"]][tid] = obs

    pair_stats = defaultdict(lambda: {"events":0, "follow":0, "stable":0, "dists":[], "same_direction":[]})
    times = sorted(by_time.keys())
    prev_by_tid = {}
    for t in times:
        obs = by_time[t]
        tids = list(obs.keys())
        for i in range(len(tids)):
            for j in range(i+1, len(tids)):
                a, b = tids[i], tids[j]
                ca, cb = tuple(obs[a]["center"]), tuple(obs[b]["center"])
                d = dist(ca, cb)
                if not (FOLLOW_DISTANCE_MIN <= d <= FOLLOW_DISTANCE_MAX):
                    continue
                total_pair_events += 1
                key = tuple(sorted((a,b)))
                pair_stats[key]["events"] += 1
                pair_stats[key]["dists"].append(d)
                if a in prev_by_tid and b in prev_by_tid:
                    va = vec(tuple(prev_by_tid[a]["center"]), ca)
                    vb = vec(tuple(prev_by_tid[b]["center"]), cb)
                    cosv = cosine(va, vb)
                    pair_stats[key]["same_direction"].append(cosv)
                    if cosv >= FOLLOW_DIRECTION_COS_THRESHOLD and norm(va) > 5 and norm(vb) > 5:
                        follow_events += 1
                        pair_stats[key]["follow"] += 1
        prev_by_tid = obs

    for key, st in pair_stats.items():
        if len(st["dists"]) >= 3:
            if np.std(st["dists"]) <= FOLLOW_DISTANCE_STABILITY:
                stable_events += st["events"]
                st["stable"] = st["events"]
        event_ratio = st["events"] / max(1, len(times))
        follow_ratio = st["follow"] / max(1, st["events"])
        stability_bonus = 0.20 if st["stable"] > 0 else 0.0
        # 스토킹은 '오랫동안 같은 방향 + 거리 유지'를 더 중시한다.
        score = 0.50 * follow_ratio + 0.25 * min(1.0, event_ratio) + stability_bonus
        if st["events"] < MIN_TRACK_LEN_FOR_STALKING:
            score *= 0.70
        if score > best_pair_score:
            best_pair_score = score
            best_pair = {
                "pair": list(key),
                "events": st["events"],
                "follow": st["follow"],
                "stable": st["stable"],
                "dist_mean": round(float(np.mean(st["dists"])), 3) if st["dists"] else None,
                "dist_std": round(float(np.std(st["dists"])), 3) if st["dists"] else None,
                "score": round(float(score), 4),
            }

    score = 0.0
    score += min(0.60, best_pair_score)
    score += min(0.20, follow_events / max(1, total_pair_events) * 0.20)
    score += min(0.10, stable_events / max(1, total_pair_events) * 0.10)
    score += min(0.10, long_track_bonus * 0.04)
    score = round(float(min(1.0, score)), 4)

    return {
        "stalking_score": score,
        "is_stalking": score >= STALKING_THRESHOLD,
        "follow_events": follow_events,
        "total_pair_events": total_pair_events,
        "stable_events": stable_events,
        "long_track_bonus": long_track_bonus,
        "best_pair": best_pair,
    }


def window_iter(frames, window_seconds=5.0, stride_seconds=1.5):
    if not frames:
        return []
    start = frames[0].get("time_sec", 0)
    end = frames[-1].get("time_sec", start)
    wins = []
    t = start
    while t <= end:
        w_frames = [fr for fr in frames if t <= fr.get("time_sec", 0) < t + window_seconds]
        if w_frames:
            wins.append((t, t + window_seconds, w_frames))
        t += stride_seconds
    return wins


def compute_fight_window_score(w_frames):
    close_pairs = 0
    multi_person = 0
    overlap_events = 0
    reversal_events = 0
    pose_vals = []
    flow_vals = []
    aggressive_motion = 0

    prev_centers_sorted = None
    for fr in w_frames:
        persons = fr.get("persons", [])
        if len(persons) >= 2:
            multi_person += 1
            centers = [tuple(p["center"]) for p in persons]
            for i in range(len(persons)):
                for j in range(i+1, len(persons)):
                    d = dist(centers[i], centers[j])
                    if d < 180:
                        close_pairs += 1
                    if bbox_iou(persons[i]["bbox"], persons[j]["bbox"]) > 0.02:
                        overlap_events += 1
            centers_sorted = sorted(centers, key=lambda x: x[0])[:2]
            if prev_centers_sorted is not None and len(centers_sorted) >= 2:
                prev_gap = prev_centers_sorted[1][0] - prev_centers_sorted[0][0]
                curr_gap = centers_sorted[1][0] - centers_sorted[0][0]
                if prev_gap * curr_gap < 0:
                    reversal_events += 1
            prev_centers_sorted = centers_sorted
        pose_vals.append(fr.get("pose_agitation", 0.0))
        fl = fr.get("flow", {})
        flow_vals.append(fl.get("p90_mag", 0.0))
        if fl.get("strong_ratio", 0.0) > 0.08:
            aggressive_motion += 1

    n = max(1, len(w_frames))
    multi_ratio = multi_person / n
    close_ratio = min(1.0, close_pairs / max(1, n))
    overlap_ratio = min(1.0, overlap_events / max(1, n))
    pose_score = min(1.0, float(np.mean(pose_vals)) / 2.8) if pose_vals else 0.0
    flow_score = min(1.0, float(np.mean(flow_vals)) / 8.0) if flow_vals else 0.0
    aggressive_ratio = aggressive_motion / n
    reversal_ratio = min(1.0, reversal_events / max(1, n))

    score = 0.0
    score += 0.10 * multi_ratio
    score += 0.22 * close_ratio
    score += 0.14 * overlap_ratio
    score += 0.18 * pose_score
    score += 0.22 * flow_score
    score += 0.10 * aggressive_ratio
    score += 0.04 * reversal_ratio

    if close_ratio < 0.08:
        score *= 0.55
    if pose_model is None:
        score += 0.05 * close_ratio + 0.05 * aggressive_ratio

    return {
        "score": round(float(min(1.0, score)), 4),
        "multi_ratio": round(multi_ratio, 4),
        "close_ratio": round(close_ratio, 4),
        "overlap_ratio": round(overlap_ratio, 4),
        "pose_score": round(pose_score, 4),
        "flow_score": round(flow_score, 4),
        "aggressive_ratio": round(aggressive_ratio, 4),
        "reversal_ratio": round(reversal_ratio, 4),
    }


def compute_fight_score(frames):
    windows = []
    best = None
    for s, e, w_frames in window_iter(frames, WINDOW_SECONDS, WINDOW_STRIDE_SECONDS):
        st = compute_fight_window_score(w_frames)
        item = {"start_time": round(s,2), "end_time": round(e,2), **st}
        windows.append(item)
        if best is None or item["score"] > best["score"]:
            best = item
    peak = best["score"] if best else 0.0
    return {
        "fight_score": peak,
        "is_fight": peak >= FIGHT_THRESHOLD,
        "best_fight_window": best,
        "windows": windows,
    }


def compute_riding_score(frames):
    tw_frames = 0
    rider_frames = 0
    sidewalk_ratios = []
    road_ratios = []
    sidewalk_band_ratios = []
    road_band_ratios = []
    seg_used_count = 0
    world_tw_count = 0
    base_tw_count = 0

    for fr in frames:
        tw = fr.get("two_wheelers", [])
        if tw:
            tw_frames += 1
        if fr.get("rider_pairs"):
            rider_frames += 1
        for obj in tw:
            cls = str(obj.get("class", "")).lower()
            if cls in TWO_WHEELER_CLASSES:
                base_tw_count += 1
            else:
                world_tw_count += 1
        if fr.get("segformer_used"):
            seg_used_count += 1
        for ev in fr.get("sidewalk_evidence", []):
            sr = ev.get("sidewalk_ratio")
            rr = ev.get("road_ratio")
            sbr = ev.get("sidewalk_band_ratio")
            rbr = ev.get("road_band_ratio")
            if sr is not None:
                sidewalk_ratios.append(sr)
            if rr is not None:
                road_ratios.append(rr)
            if sbr is not None:
                sidewalk_band_ratios.append(sbr)
            if rbr is not None:
                road_band_ratios.append(rbr)

    n = max(1, len(frames))
    two_wheeler_ratio = tw_frames / n
    rider_ratio = rider_frames / n
    sidewalk_mean = float(np.mean(sidewalk_ratios)) if sidewalk_ratios else 0.0
    road_mean = float(np.mean(road_ratios)) if road_ratios else 0.0
    sidewalk_peak = float(np.max(sidewalk_ratios)) if sidewalk_ratios else 0.0
    sidewalk_band_mean = float(np.mean(sidewalk_band_ratios)) if sidewalk_band_ratios else 0.0
    road_band_mean = float(np.mean(road_band_ratios)) if road_band_ratios else 0.0

    detection_score = min(1.0, two_wheeler_ratio * 0.65 + rider_ratio * 0.55 + min(0.25, world_tw_count * 0.012))
    sidewalk_score = min(1.0, sidewalk_mean * 0.45 + sidewalk_peak * 0.25 + sidewalk_band_mean * 0.30)

    if detection_score < 0.08:
        riding_label = "no_two_wheeler"
    elif sidewalk_score >= RIDE_SIDEWALK_THRESHOLD and sidewalk_mean + sidewalk_band_mean * 0.5 >= road_mean - 0.05:
        riding_label = "two_wheeler_sidewalk_riding"
    elif sidewalk_score >= RIDE_SIDEWALK_CANDIDATE_THRESHOLD:
        riding_label = "two_wheeler_sidewalk_candidate"
    else:
        riding_label = "two_wheeler_rider_like"

    return {
        "riding_label": riding_label,
        "two_wheeler_detected": detection_score >= 0.08,
        "two_wheeler_ratio": round(two_wheeler_ratio, 4),
        "rider_ratio": round(rider_ratio, 4),
        "detection_score": round(detection_score, 4),
        "sidewalk_score": round(sidewalk_score, 4),
        "sidewalk_mean": round(sidewalk_mean, 4),
        "sidewalk_peak": round(sidewalk_peak, 4),
        "sidewalk_band_mean": round(sidewalk_band_mean, 4),
        "road_mean": round(road_mean, 4),
        "road_band_mean": round(road_band_mean, 4),
        "segformer_frame_count": seg_used_count,
        "base_two_wheeler_object_count": base_tw_count,
        "world_two_wheeler_object_count": world_tw_count,
    }


def compute_helmet_score(frames):
    rider_pair_frames = 0
    helmet_checked_frames = 0
    no_helmet_hits = 0
    helmet_hits = 0
    unknown_hits = 0
    evidence = []

    for fr in frames:
        if fr.get("rider_pairs"):
            rider_pair_frames += 1
        hev = fr.get("helmet_evidence", [])
        if hev:
            helmet_checked_frames += 1
        for ev in hev:
            st = ev.get("helmet_status", "unknown")
            if st == "no_helmet":
                no_helmet_hits += 1
                evidence.append({"frame_id": fr.get("frame_id"), "time_sec": fr.get("time_sec"), **ev})
            elif st == "helmet":
                helmet_hits += 1
            else:
                unknown_hits += 1

    checked = max(1, no_helmet_hits + helmet_hits + unknown_hits)
    no_ratio = no_helmet_hits / checked
    helmet_ratio = helmet_hits / checked

    if helmet_backend == "none":
        label = "helmet_model_not_connected"
        is_no_helmet = False
    elif no_helmet_hits >= 2 or (no_helmet_hits >= 1 and no_ratio >= NO_HELMET_THRESHOLD):
        label = "no_helmet"
        is_no_helmet = True
    elif helmet_hits > 0 and no_helmet_hits == 0:
        label = "helmet"
        is_no_helmet = False
    elif rider_pair_frames > 0:
        label = "helmet_unknown"
        is_no_helmet = False
    else:
        label = "no_rider_for_helmet_check"
        is_no_helmet = False

    helmet_check_ratio = helmet_checked_frames / max(1, rider_pair_frames)

    return {
        "helmet_label": label,
        "is_no_helmet": is_no_helmet,
        "helmet_backend": helmet_backend,
        "rider_pair_frames": rider_pair_frames,
        "helmet_checked_frames": helmet_checked_frames,
        "helmet_check_ratio": round(helmet_check_ratio, 4),
        "no_helmet_hits": no_helmet_hits,
        "helmet_hits": helmet_hits,
        "unknown_hits": unknown_hits,
        "no_helmet_ratio": round(no_ratio, 4),
        "helmet_ratio": round(helmet_ratio, 4),
        "evidence": evidence[:20],
    }


def build_candidate_scores(fight, stalking, riding, helmet):
    """
    영상 하나에 여러 후보가 동시에 잡혀도, final_label은 가장 강한 후보 하나만 선택한다.
    candidate_scores에는 보조 후보를 남겨서 왜 그렇게 판단했는지 확인한다.
    """
    candidates = []

    fs = float(fight.get("fight_score", 0.0))
    ss = float(stalking.get("stalking_score", 0.0))
    det = float(riding.get("detection_score", 0.0))
    sidewalk = float(riding.get("sidewalk_score", 0.0))
    rl = riding.get("riding_label", "no_two_wheeler")
    no_hits = int(helmet.get("no_helmet_hits", 0))
    no_ratio = float(helmet.get("no_helmet_ratio", 0.0))

    # 싸움/스토킹은 threshold를 넘은 경우에만 강한 후보로 본다.
    candidates.append({
        "label": "fight",
        "score": round(fs * (1.20 if fight.get("is_fight") else 0.35), 4),
        "raw_score": round(fs, 4),
        "active": bool(fight.get("is_fight")),
        "reason": "fight_score"
    })
    candidates.append({
        "label": "stalking",
        "score": round(ss * (1.10 if stalking.get("is_stalking") else 0.35), 4),
        "raw_score": round(ss, 4),
        "active": bool(stalking.get("is_stalking")),
        "reason": "stalking_score"
    })

    # 인도주행은 단순 이륜 검출보다 sidewalk 증거에 가중치를 준다.
    if rl == "two_wheeler_sidewalk_riding":
        sc = sidewalk * 1.25 + det * 0.20
        active = True
    elif rl == "two_wheeler_sidewalk_candidate":
        sc = sidewalk * 1.00 + det * 0.12
        active = True
    elif rl == "two_wheeler_rider_like":
        # 실제 이상행동 확정이라기보다는 보조 후보. final에서는 강하게 밀지 않는다.
        sc = det * 0.42
        active = False
    else:
        sc = 0.0
        active = False
    if rl != "no_two_wheeler":
        candidates.append({
            "label": rl,
            "score": round(float(sc), 4),
            "raw_score": round(sidewalk, 4),
            "active": active,
            "reason": f"riding_label={rl}, detection={det:.3f}, sidewalk={sidewalk:.3f}"
        })

    # 헬멧은 no_helmet이 나온 경우에만 최종 후보로 강하게 반영한다.
    if helmet.get("helmet_label") == "no_helmet":
        sc = min(1.0, no_ratio * 0.75 + min(1.0, no_hits / 3.0) * 0.45)
        candidates.append({
            "label": "no_helmet",
            "score": round(float(sc), 4),
            "raw_score": round(no_ratio, 4),
            "active": True,
            "reason": f"no_helmet_hits={no_hits}, no_helmet_ratio={no_ratio:.3f}"
        })
    elif helmet.get("helmet_label") == "helmet_unknown" and det >= 0.15:
        candidates.append({
            "label": "helmet_unknown",
            "score": round(det * 0.20, 4),
            "raw_score": 0.0,
            "active": False,
            "reason": "rider exists but helmet model was inconclusive"
        })

    candidates.sort(key=lambda x: x["score"], reverse=True)
    return candidates


def decide_primary_label(candidate_scores):
    if not candidate_scores:
        return "other", []
    top = candidate_scores[0]
    active_candidates = [c for c in candidate_scores if c.get("active") and c.get("score", 0) >= PRIMARY_LABEL_MIN_SCORE]
    if not active_candidates:
        return "other", candidate_scores
    active_candidates.sort(key=lambda x: x["score"], reverse=True)
    top = active_candidates[0]
    return top["label"], candidate_scores


def make_candidate_tags(primary_label, candidate_scores):
    tags = [primary_label if primary_label else "other"]
    for c in candidate_scores:
        if c.get("active") or c.get("score", 0) >= PRIMARY_LABEL_MIN_SCORE:
            lab = c.get("label")
            if lab and lab not in tags:
                tags.append(lab)
    return tags


In [9]:
# ============================================================
# Cell 7. AIO 최종 판정 실행  [helmet_label 포함]
# ============================================================
feature_json_paths = sorted(AIO_FEATURE_DIR.rglob("*.json"))
print("feature json 수:", len(feature_json_paths))

results = []
failed = []

for jp in tqdm(feature_json_paths, desc="AIO final classify"):
    try:
        with open(jp, "r", encoding="utf-8") as f:
            data = json.load(f)
        frames = data.get("frames", [])
        tracks = assign_tracks(frames)
        stalking = compute_stalking_score(frames, tracks)
        fight = compute_fight_score(frames)
        riding = compute_riding_score(frames)
        helmet = compute_helmet_score(frames)
        candidate_scores = build_candidate_scores(fight, stalking, riding, helmet)
        final_label, candidate_scores = decide_primary_label(candidate_scores)
        candidate_tags = make_candidate_tags(final_label, candidate_scores)
        action_label = final_label if final_label in {"fight", "stalking"} else "other"
        final_tags = candidate_tags

        track_summary = {
            str(tid): {**track_motion(tr), "first_time": tr[0]["time_sec"], "last_time": tr[-1]["time_sec"]}
            for tid, tr in tracks.items() if tr
        }

        item = {
            "video_name": data.get("video_name"),
            "video_path": data.get("video_path"),
            "relative_video_path": data.get("relative_video_path"),
            "final_label": final_label,
            "final_tags": final_tags,
            "candidate_tags": candidate_tags,
            "candidate_scores": candidate_scores,
            "action_label": action_label,
            "fight_label": "fight" if fight.get("is_fight") else "not_fight",
            "stalking_label": "stalking" if stalking.get("is_stalking") else "not_stalking",
            "riding_label": riding.get("riding_label"),
            "helmet_label": helmet.get("helmet_label"),
            "fight": {k:v for k,v in fight.items() if k != "windows"},
            "stalking": stalking,
            "riding": riding,
            "helmet": {k:v for k,v in helmet.items() if k != "evidence"},
            "helmet_evidence": helmet.get("evidence", []),
            "track_summary": track_summary,
            "model_status": MODEL_STATUS,
            "feature_json": str(jp),
        }
        results.append(item)
    except Exception as e:
        import traceback
        failed.append({"json": str(jp), "error": repr(e), "traceback": traceback.format_exc()})
        print("\nFAILED JSON:", jp)
        traceback.print_exc()

summary = {
    "task": "aio_multi_model_parallel_classification_improved",
    "aio_version": AIO_VERSION,
    "video_root": str(VIDEO_ROOT),
    "output_dir": str(AIO_DIR),
    "feature_dir": str(AIO_FEATURE_DIR),
    "model_status": MODEL_STATUS,
    "settings": {
        "frame_interval": FRAME_INTERVAL,
        "fight_threshold": FIGHT_THRESHOLD,
        "stalking_threshold": STALKING_THRESHOLD,
        "ride_sidewalk_threshold": RIDE_SIDEWALK_THRESHOLD,
        "no_helmet_threshold": NO_HELMET_THRESHOLD,
        "primary_label_min_score": PRIMARY_LABEL_MIN_SCORE,
        "primary_label_margin": PRIMARY_LABEL_MARGIN,
        "window_seconds": WINDOW_SECONDS,
        "window_stride_seconds": WINDOW_STRIDE_SECONDS,
    },
    "total_json": len(feature_json_paths),
    "success_count": len(results),
    "failed_count": len(failed),
    "final_label_count": dict(Counter(r["final_label"] for r in results)),
    "action_label_count": dict(Counter(r["action_label"] for r in results)),
    "riding_label_count": dict(Counter(r["riding_label"] for r in results)),
    "helmet_label_count": dict(Counter(r["helmet_label"] for r in results)),
    "results": results,
    "failed": failed,
}

summary_path = AIO_FINAL_DIR / "aio_final_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("="*80)
print("AIO 최종 판정 완료")
print("성공:", len(results), "실패:", len(failed))
print("summary 저장:", summary_path)
print("final_label_count:", summary["final_label_count"])
print("action_label_count:", summary["action_label_count"])
print("riding_label_count:", summary["riding_label_count"])
print("helmet_label_count:", summary["helmet_label_count"])
if failed:
    for x in failed[:10]:
        print(x)


feature json 수: 22


AIO final classify: 100%|██████████████████████████████████████████████████████████████| 22/22 [00:00<00:00, 86.07it/s]

AIO 최종 판정 완료
성공: 22 실패: 0
summary 저장: C:\Users\우성한\outputs\prediction\qq\final_aio\aio_final_summary.json
final_label_count: {'stalking': 2, 'other': 14, 'two_wheeler_sidewalk_candidate': 3, 'two_wheeler_sidewalk_riding': 3}
action_label_count: {'stalking': 2, 'other': 20}
riding_label_count: {'two_wheeler_rider_like': 16, 'two_wheeler_sidewalk_candidate': 3, 'two_wheeler_sidewalk_riding': 3}
helmet_label_count: {'helmet_model_not_connected': 22}


In [10]:
# ============================================================
# Cell 8. 결과를 보기 쉽게 출력 + txt 저장  [primary label 중심]
# ============================================================
summary_path = AIO_FINAL_DIR / "aio_final_summary.json"
with open(summary_path, "r", encoding="utf-8") as f:
    summary = json.load(f)

results = summary.get("results", [])
print("="*100)
print("QQ 결과 요약")
print("total:", len(results))
print("final_label_count:", summary.get("final_label_count"))
print("action_label_count:", summary.get("action_label_count"))
print("riding_label_count:", summary.get("riding_label_count"))
print("helmet_label_count:", summary.get("helmet_label_count"))
print("model_status.helmet_backend:", summary.get("model_status", {}).get("helmet_backend"))
print("model_status.world_model_active:", summary.get("model_status", {}).get("world_model_active"))


def _short_path(p):
    if not p:
        return ""
    parts = str(p).replace("\\", "/").split("/")
    return "/".join(parts[-4:]) if len(parts) >= 4 else str(p)


def _top_candidates(r, k=4):
    arr = r.get("candidate_scores", [])[:k]
    return ", ".join([f"{c.get('label')}:{c.get('score')}" for c in arr])

print("\n===== 영상별 최종 판정표 =====")
header = f"{'idx':>3} | {'final':<32} | {'fight':>6} | {'stalk':>6} | {'ride':<32} | {'helmet':<24} | video"
print(header)
print("-"*len(header))
for i, r in enumerate(results, start=1):
    print(
        f"{i:>3} | "
        f"{r.get('final_label',''):<32} | "
        f"{r.get('fight',{}).get('fight_score',0):>6} | "
        f"{r.get('stalking',{}).get('stalking_score',0):>6} | "
        f"{r.get('riding_label',''):<32} | "
        f"{r.get('helmet_label',''):<24} | "
        f"{r.get('video_name')}"
    )
    print(f"      candidates: {_top_candidates(r)}")
    print(f"      path: {_short_path(r.get('relative_video_path'))}")

print("\n===== final label별 영상 =====")
by_final = defaultdict(list)
for r in results:
    by_final[r.get("final_label", "other")].append(r.get("video_name"))
for lab, vids in sorted(by_final.items(), key=lambda x: (-len(x[1]), x[0])):
    print(f"{lab}: {len(vids)}")
    for v in vids[:20]:
        print("  -", v)

print("\n===== no helmet 후보 TOP 20 =====")
for r in sorted(results, key=lambda x: x.get("helmet", {}).get("no_helmet_ratio", 0) + x.get("helmet", {}).get("no_helmet_hits", 0), reverse=True)[:20]:
    print({
        "video": r.get("video_name"),
        "final": r.get("final_label"),
        "helmet_label": r.get("helmet_label"),
        "backend": r.get("helmet", {}).get("helmet_backend"),
        "checked_frames": r.get("helmet", {}).get("helmet_checked_frames"),
        "no_helmet_hits": r.get("helmet", {}).get("no_helmet_hits"),
        "helmet_hits": r.get("helmet", {}).get("helmet_hits"),
        "unknown_hits": r.get("helmet", {}).get("unknown_hits"),
        "ratio": r.get("helmet", {}).get("no_helmet_ratio"),
        "candidates": r.get("candidate_scores", [])[:3],
    })

print("\n===== fight TOP 10 =====")
for r in sorted(results, key=lambda x: x.get("fight", {}).get("fight_score", 0), reverse=True)[:10]:
    print({"video": r.get("video_name"), "final": r.get("final_label"), "score": r.get("fight", {}).get("fight_score"), "candidate": r.get("candidate_scores", [])[:3]})

print("\n===== stalking TOP 10 =====")
for r in sorted(results, key=lambda x: x.get("stalking", {}).get("stalking_score", 0), reverse=True)[:10]:
    print({"video": r.get("video_name"), "final": r.get("final_label"), "score": r.get("stalking", {}).get("stalking_score"), "candidate": r.get("candidate_scores", [])[:3]})

print("\n===== riding / sidewalk TOP 10 =====")
for r in sorted(results, key=lambda x: x.get("riding", {}).get("sidewalk_score", 0) + x.get("riding", {}).get("detection_score", 0) * 0.2, reverse=True)[:10]:
    print({
        "video": r.get("video_name"),
        "final": r.get("final_label"),
        "riding_label": r.get("riding_label"),
        "detection_score": r.get("riding", {}).get("detection_score"),
        "sidewalk_score": r.get("riding", {}).get("sidewalk_score"),
        "world_tw_count": r.get("riding", {}).get("world_two_wheeler_object_count"),
        "base_tw_count": r.get("riding", {}).get("base_two_wheeler_object_count"),
        "candidate": r.get("candidate_scores", [])[:3],
    })

report_path = AIO_FINAL_DIR / "qq_readable_report.txt"
with open(report_path, "w", encoding="utf-8") as f:
    f.write("QQ 결과 요약\n")
    f.write(f"total: {len(results)}\n")
    f.write(f"final_label_count: {summary.get('final_label_count')}\n")
    f.write(f"action_label_count: {summary.get('action_label_count')}\n")
    f.write(f"riding_label_count: {summary.get('riding_label_count')}\n")
    f.write(f"helmet_label_count: {summary.get('helmet_label_count')}\n")
    f.write(f"helmet_backend: {summary.get('model_status', {}).get('helmet_backend')}\n")
    f.write("\n영상별 최종 판정\n")
    for i, r in enumerate(results, start=1):
        f.write(f"[{i}] {r.get('video_name')}\n")
        f.write(f"  final_label: {r.get('final_label')}\n")
        f.write(f"  candidates: {_top_candidates(r, 6)}\n")
        f.write(f"  fight_score: {r.get('fight',{}).get('fight_score')}\n")
        f.write(f"  stalking_score: {r.get('stalking',{}).get('stalking_score')}\n")
        f.write(f"  riding_label: {r.get('riding_label')} / detection={r.get('riding',{}).get('detection_score')} / sidewalk={r.get('riding',{}).get('sidewalk_score')}\n")
        f.write(f"  helmet_label: {r.get('helmet_label')} / checked={r.get('helmet',{}).get('helmet_checked_frames')} / no_hits={r.get('helmet',{}).get('no_helmet_hits')} / helmet_hits={r.get('helmet',{}).get('helmet_hits')}\n")
        f.write(f"  path: {r.get('relative_video_path')}\n\n")
print("report 저장:", report_path)


QQ 결과 요약
total: 22
final_label_count: {'stalking': 2, 'other': 14, 'two_wheeler_sidewalk_candidate': 3, 'two_wheeler_sidewalk_riding': 3}
action_label_count: {'stalking': 2, 'other': 20}
riding_label_count: {'two_wheeler_rider_like': 16, 'two_wheeler_sidewalk_candidate': 3, 'two_wheeler_sidewalk_riding': 3}
helmet_label_count: {'helmet_model_not_connected': 22}
model_status.helmet_backend: none
model_status.world_model_active: True

===== 영상별 최종 판정표 =====
idx | final                            |  fight |  stalk | ride                             | helmet                   | video
------------------------------------------------------------------------------------------------------------------------------
  1 | stalking                         | 0.0913 | 0.6675 | two_wheeler_rider_like           | helmet_model_not_connected | lo_e1281_c1.mp4
      candidates: stalking:0.7343, two_wheeler_rider_like:0.42, fight:0.032
      path: 스토킹/특정 구역 내 지속 배회/lo_e1281/lo_e1281_c1.mp4
  2 | other     

In [11]:
# ============================================================
# Cell 9. 자연어 비슷한 검색 함수  [헬멧 포함]
# ============================================================
def search_aio_results(query, top_k=50):
    q = str(query).lower()
    wants_fight = any(k in q for k in ["싸움", "폭행", "fight", "violence", "충돌"])
    wants_stalking = any(k in q for k in ["스토킹", "따라", "추적", "배회", "stalking", "follow"])
    wants_sidewalk = any(k in q for k in ["인도", "보도", "sidewalk", "침범"])
    wants_tw = any(k in q for k in ["오토바이", "자전거", "킥보드", "스쿠터", "이륜", "motorcycle", "bicycle", "scooter", "kickboard"])
    wants_no_helmet = any(k in q for k in ["헬멧", "노헬멧", "미착용", "no helmet", "no_helmet", "without helmet"])
    wants_unknown_helmet = any(k in q for k in ["헬멧 불명", "helmet unknown", "helmet_unknown"])

    hits = []
    for r in results:
        ok = False
        score = 0.0
        if wants_fight and r.get("fight_label") == "fight":
            ok = True; score += r.get("fight", {}).get("fight_score", 0) + 1
        if wants_stalking and r.get("stalking_label") == "stalking":
            ok = True; score += r.get("stalking", {}).get("stalking_score", 0) + 1
        if wants_sidewalk and r.get("riding_label") in {"two_wheeler_sidewalk_riding", "two_wheeler_sidewalk_candidate"}:
            ok = True; score += r.get("riding", {}).get("sidewalk_score", 0) + 1
        if wants_tw and r.get("riding_label") != "no_two_wheeler":
            ok = True; score += r.get("riding", {}).get("detection_score", 0) + 1
        if wants_no_helmet and r.get("helmet_label") == "no_helmet":
            ok = True; score += r.get("helmet", {}).get("no_helmet_ratio", 0) + 1
        if wants_unknown_helmet and r.get("helmet_label") == "helmet_unknown":
            ok = True; score += 1
        if not any([wants_fight, wants_stalking, wants_sidewalk, wants_tw, wants_no_helmet, wants_unknown_helmet]):
            text = " ".join([r.get("final_label", ""), " ".join(r.get("candidate_tags", r.get("final_tags", [])))]).lower()
            if q in text:
                ok = True; score += 1
        if ok:
            item = dict(r)
            item["search_score"] = round(score, 4)
            hits.append(item)

    hits = sorted(hits, key=lambda x: x.get("search_score", 0), reverse=True)
    print("QUERY:", query)
    print("hits:", len(hits))
    for h in hits[:top_k]:
        print({
            "video_name": h.get("video_name"),
            "final_label": h.get("final_label"),
            "fight_score": h.get("fight", {}).get("fight_score"),
            "stalking_score": h.get("stalking", {}).get("stalking_score"),
            "riding_label": h.get("riding_label"),
            "sidewalk_score": h.get("riding", {}).get("sidewalk_score"),
            "helmet_label": h.get("helmet_label"),
            "no_helmet_hits": h.get("helmet", {}).get("no_helmet_hits"),
            "search_score": h.get("search_score"),
        })
    return hits

# 예시
# search_aio_results("싸움")
# search_aio_results("스토킹")
# search_aio_results("인도 주행 오토바이")
# search_aio_results("헬멧 미착용")
# search_aio_results("킥보드")
